# 🚀 WORKER COLAB: TỰ ĐỘNG SINH VIDEO LTX-VIDEO (CHỐNG TRÀN RAM & VRAM)
Sổ tay này chạy theo kiến trúc **Two-Phase Pre-Encoding**:
1. **Pha 1:** Nạp riêng T5 text encoder (8-bit hoặc float16) mã hóa prompt thành `prompt_embeds.pt` rồi **xóa khỏi RAM**.
2. **Pha 2:** Nạp pipeline LTX-Video (`text_encoder=None`) sinh clip 704x384 @ 24fps trên Tesla T4 an toàn tuyệt đối.
3. Tự động đọc từ `jobs/pending/`, xuất sang `jobs/done/` và Google Drive.

In [ ]:
#@title ⚙️ 1. Cài đặt môi trường tối ưu cho Tesla T4
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q "diffusers>=0.31.0" "transformers>=4.45.0" "accelerate>=0.34.0" bitsandbytes sentencepiece psutil

import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/(1024**3):.2f} GB trống / {total/(1024**3):.2f} GB tổng")

In [ ]:
#@title 📂 2. Kết nối Google Drive & Clone / Pull Codebase
from google.colab import drive
import os

# Gắn kết Google Drive để bảo toàn dữ liệu khi runtime ngắt
drive.mount('/content/drive')

WORKSPACE_DIR = "/content/Create-Video"
if not os.path.exists(WORKSPACE_DIR):
    !git clone https://github.com/SieuHuynh-glitch/Create-Video.git {WORKSPACE_DIR}
else:
    %cd {WORKSPACE_DIR}
    !git pull origin master

%cd {WORKSPACE_DIR}

In [ ]:
#@title 🤖 3. Chạy Headless Worker xử lý hàng đợi (Tự động phục hồi)
# Lệnh này sẽ xử lý toàn bộ jobs/pending/natural-cycle và xuất ra jobs/done/natural-cycle
!python scripts/colab_worker.py --slug natural-cycle